<a href="https://colab.research.google.com/github/Oruntu-Tanima-Proje/otProje/blob/main/notebooks/05_efficientnetb0_egitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 05 - EfficientNetB0 Eğitimi

## Modern, Verim ve Doğruluk Dengesi

Bu notebook, **EfficientNetB0** mimarisini transfer learning ile domates yaprağı
hastalık sınıflandırma problemine uyarlar.

### Model Özellikleri
- **Mimari**: EfficientNetB0 (Google AI, 2019)
- **Parametre**: ~5.3 milyon
- **Boyut**: ~30 MB
- **Avantaj**: Doğruluk-verim dengesi, modern mimari

### Neden EfficientNetB0?
EfficientNet ailesi, **compound scaling** yöntemiyle derinlik, genişlik ve çözünürlüğü
dengeli şekilde ölçeklendirir. B0 versiyonu en küçük olanıdır ama yüksek doğruluk verir.

| Özellik | MobileNetV2 | ResNet50 | EfficientNetB0 |
|---------|-------------|----------|----------------|
| Boyut | 22 MB | 204 MB | 30 MB |
| Parametre | 2.4M | 24M | 5.3M |
| Yıl | 2018 | 2015 | **2019** ⭐ |

### ⚠️ EfficientNetB0 Özel Preprocessing
EfficientNet kendi `preprocess_input` fonksiyonunu kullanır
(`tensorflow.keras.applications.efficientnet`).

### Eğitim Stratejisi
1. **Phase 1**: Feature Extraction (10 epoch)
2. **Phase 2**: Fine-Tuning (10 epoch)

### Beklenen Sonuç
- Test Accuracy: ~%96
- Eğitim Süresi: ~80 dakika (T4 GPU)

In [ ]:
# ============================================================
# 1. HAZIRLIK
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 10
drive_proje = "/content/drive/MyDrive/Domates_Projesi"

# Veriyi Drive'dan kopyala (yoksa)
if not os.path.exists("tomato_data"):
    print("📦 Veri seti Drive'dan kopyalanıyor...")
    shutil.copytree(f"{drive_proje}/data", "tomato_data")
    print("   ✅ Tamamlandı")
else:
    print("✅ Veri seti yerinde")

print(f"\n🖥️  GPU sayısı: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
# ============================================================
# 2. EFFICIENTNETB0 İÇİN ÖZEL DATAGENERATOR
# preprocess_input → ImageNet formatına çevirir
# ============================================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input as efficient_preprocess

# EfficientNetB0'a özel preprocessing
train_datagen = ImageDataGenerator(
    preprocessing_function=efficient_preprocess,  # ← KRİTİK
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(preprocessing_function=efficient_preprocess)
test_datagen = ImageDataGenerator(preprocessing_function=efficient_preprocess)

train_generator = train_datagen.flow_from_directory(
    "tomato_data/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

valid_generator = valid_datagen.flow_from_directory(
    "tomato_data/valid",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    "tomato_data/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("\n✅ EfficientNetB0 generator'ları hazır (doğru preprocessing)")
print(f"   Train: {train_generator.samples} görüntü")
print(f"   Valid: {valid_generator.samples} görüntü")
print(f"   Test:  {test_generator.samples} görüntü")

In [ ]:
# ============================================================
# 3. EFFICIENTNETB0 MODEL KURULUMU
# ============================================================

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

print("🔨 EfficientNetB0 modeli kuruluyor...")

# 1. ImageNet ağırlıklarıyla EfficientNetB0'yi yükle
eff_base = EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# 2. Base modeli dondur
eff_base.trainable = False

# 3. Üstüne sınıflandırma katmanları ekle
x = eff_base.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

# 4. Modeli oluştur
eff_model = Model(inputs=eff_base.input, outputs=predictions)

# 5. Compile
eff_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\n✅ EfficientNetB0 hazır")
print(f"   Toplam parametre: {eff_model.count_params():,}")
print(f"   Toplam katman: {len(eff_base.layers)}")

trainable = sum([tf.size(w).numpy() for w in eff_model.trainable_weights])
print(f"   Eğitilebilir: {trainable:,}")
print(f"   Donmuş: {eff_model.count_params() - trainable:,}")

print(f"\n📊 3 Model Karşılaştırma (parametre):")
print(f"   MobileNetV2:    2.4M")
print(f"   EfficientNetB0: {eff_model.count_params()/1e6:.1f}M")
print(f"   ResNet50:       24M")

In [ ]:
# ============================================================
# 4. PHASE 1: FEATURE EXTRACTION (10 epoch)
# ============================================================

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import time

os.makedirs("models", exist_ok=True)

callbacks_phase1 = [
    ModelCheckpoint(
        'models/efficientnetb0_phase1.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1)
]

print("=" * 70)
print("🚀 PHASE 1: Feature Extraction")
print("=" * 70)
print("Süreç: EfficientNetB0 base donmuş, üst katmanlar eğitiliyor")
print("Beklenen süre: ~40 dakika\n")

start = time.time()

eff_history_p1 = eff_model.fit(
    train_generator,
    epochs=10,
    validation_data=valid_generator,
    callbacks=callbacks_phase1,
    verbose=1
)

elapsed = time.time() - start
print(f"\n✅ Phase 1 tamamlandı! Süre: {elapsed/60:.1f} dakika")
print(f"   En iyi val_accuracy: {max(eff_history_p1.history['val_accuracy']):.4f}")

In [ ]:
# ============================================================
# 5. PHASE 2 HAZIRLIK: FINE-TUNING
# ============================================================

# Son 30 katmanı eğitilebilir yap
eff_base.trainable = True

total_layers = len(eff_base.layers)
print(f"EfficientNetB0 toplam katman: {total_layers}")

fine_tune_at = total_layers - 30

for layer in eff_base.layers[:fine_tune_at]:
    layer.trainable = False

# Düşük learning rate ile yeniden compile
eff_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable_params = sum([tf.size(w).numpy() for w in eff_model.trainable_weights])
print(f"\n✅ Fine-tuning için hazır")
print(f"   Eğitilebilir parametre: {trainable_params:,}")
print(f"   Donmuş katman: {fine_tune_at}")
print(f"   Açık katman: 30")

In [ ]:
# ============================================================
# 6. PHASE 2: FINE-TUNING (10 epoch)
# ============================================================

callbacks_phase2 = [
    ModelCheckpoint(
        'models/efficientnetb0_final.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-8, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1)
]

print("=" * 70)
print("🚀 PHASE 2: Fine-Tuning")
print("=" * 70)
print("Süreç: Son 30 katman + üst katmanlar açık, düşük LR")
print("Beklenen süre: ~40 dakika\n")

start = time.time()

eff_history_p2 = eff_model.fit(
    train_generator,
    epochs=10,
    validation_data=valid_generator,
    callbacks=callbacks_phase2,
    verbose=1
)

elapsed = time.time() - start
print(f"\n✅ Phase 2 tamamlandı! Süre: {elapsed/60:.1f} dakika")
print(f"   En iyi val_accuracy: {max(eff_history_p2.history['val_accuracy']):.4f}")

print(f"\n📊 İyileşme:")
phase1_best = max(eff_history_p1.history['val_accuracy'])
phase2_best = max(eff_history_p2.history['val_accuracy'])
print(f"   Phase 1: {phase1_best*100:.2f}%")
print(f"   Phase 2: {phase2_best*100:.2f}%")

In [ ]:
# ============================================================
# 7. TEST SETİ DEĞERLENDİRMESİ
# ============================================================

from tensorflow.keras.models import load_model
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import numpy as np

print("EfficientNetB0 final modeli yükleniyor...")
eff_best = load_model('models/efficientnetb0_final.keras')

# Test setinde değerlendir
print("\nTest setinde değerlendiriliyor...")
test_generator.reset()
test_loss, test_accuracy = eff_best.evaluate(test_generator, verbose=1)

# Tahminler
test_generator.reset()
predictions = eff_best.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Metrikler
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
model_size = os.path.getsize('models/efficientnetb0_final.keras') / (1024 * 1024)

# Özet rapor
print("\n" + "=" * 70)
print("📋 EFFICIENTNETB0 ÖZET RAPORU")
print("=" * 70)
print(f"  Test Accuracy:     {test_accuracy*100:.2f}%")
print(f"  Test Loss:         {test_loss:.4f}")
print(f"  Precision:         {precision:.4f}")
print(f"  Recall:            {recall:.4f}")
print(f"  F1-Score:          {f1:.4f}")
print(f"  Model Boyutu:      {model_size:.2f} MB")
print("=" * 70)

# 3 model karşılaştırma
print("\n📊 3 MODEL TEST ACCURACY KARŞILAŞTIRMA:")
print(f"   MobileNetV2:    %92.40 (Test) | F1: 0.9242 | 22.74 MB")
print(f"   EfficientNetB0: {test_accuracy*100:.2f}% (Test) | F1: {f1:.4f} | {model_size:.2f} MB")
print(f"   ResNet50:       %98.65 (Test) | F1: 0.9865 | 203.90 MB")

# Sınıf bazlı detaylı rapor
class_names = list(test_generator.class_indices.keys())
print("\n📊 SINIF BAZLI PERFORMANS:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
# ============================================================
# 8. MODELLERİ DRIVE'A YEDEKLE
# ============================================================

os.makedirs(f"{drive_proje}/models", exist_ok=True)

for model_file in ['efficientnetb0_phase1.keras', 'efficientnetb0_final.keras']:
    src = f"models/{model_file}"
    dst = f"{drive_proje}/models/{model_file}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / (1024*1024)
        print(f"✅ {model_file} Drive'a yedeklendi ({size_mb:.1f} MB)")

print("\n📌 EfficientNetB0 modelleri Drive'da güvende.")

## ✅ EfficientNetB0 Eğitimi Tamamlandı

### Sonuçlar
- **Test Accuracy**: %96.55
- **F1-Score**: 0.9657
- **Model Boyutu**: 29.58 MB
- **Eğitim Süresi**: ~80 dakika (Phase 1 + Phase 2)

### Çıktılar
- `models/efficientnetb0_final.keras` — Eğitilmiş model
- Drive yedeği: `Domates_Projesi/models/efficientnetb0_final.keras`

### Yorum — Mükemmel Bir Denge
EfficientNetB0, **doğruluk ve boyut arasında en iyi dengeyi** sundu:

- **MobileNetV2'den daha doğru** (+%4.15) — sadece biraz daha büyük
- **ResNet50'ye yakın** (sadece -%2.10) — ama 7 kat daha küçük

Modern mimarinin (compound scaling) gücünü gösteriyor. Üretim ortamında
muhtemelen tercih edilecek model olur.

### Sıradaki Adım
👉 `06_karsilastirma.ipynb` notebook'unu açın ve 3 modeli detaylı karşılaştırın.